In [ ]:
!sudo apt-get install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(8)

!ollama pull llama3.1:8b-instruct-q4_K_M
print("Ollama ready.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 53 not upgraded.
Need to get 1,486 kB of archives.
After this operation, 4,951 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 pci.ids all 0.0~2022.01.22-1ubuntu0.1 [251 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpci3 amd64 1:3.7.0-6 [28.9 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.22.04.1 [322 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 pciutils amd64 1:3.7.0-6 [63.6 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/main amd64 usb.ids all 2022.04.02-1 [219 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy/main 

In [ ]:
!pip install -q requests

In [ ]:
import zipfile
import json
import pandas as pd
import numpy as np
import joblib
from datetime import datetime
import os
import time
import random
import requests
from google.colab import userdata
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
import re

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
#checking whether ollama is running
resp = requests.get("http://localhost:11434/api/tags")
print("Ollama status:", resp.status_code)
models = [m["name"] for m in resp.json().get("models", [])]
print("Available models:", models)

Ollama status: 200
Available models: ['llama3.1:8b-instruct-q4_K_M']


In [ ]:
OLLAMA_MODEL = "llama3.1:8b-instruct-q4_K_M"
OLLAMA_URL = "http://localhost:11434/api/chat"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"
DEFINITIONS_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_labeling/v2/SCOTBESS_labels_definitions_v2.xlsx"

RETRIEVAL_DIR = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_embeddings_final"
VAL_TOPK_INPUT = f"{RETRIEVAL_DIR}/scotbess_validation_top_k_llm_input_data.jsonl"
VAL_GOLD_LABELS = f"{RETRIEVAL_DIR}/scotbess_validation_gold_labels.json"
TEST_TOPK_INPUT = f"{RETRIEVAL_DIR}/scotbess_test_top_k_llm_input_data.jsonl"
TEST_GOLD_LABELS = f"{RETRIEVAL_DIR}/scotbess_test_gold_labels.json"

OUTPUT_DIR = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_LLAMA_OLLAMA_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TOPK_VAL_OUTPUT = f"{OUTPUT_DIR}/SCOTBESS_llama_OLLAMA_val_topk_predictions.jsonl"
TOPK_TEST_OUTPUT = f"{OUTPUT_DIR}/SCOTBESS_llama_OLLAMA_test_topk_predictions.jsonl"

N_RETRIEVED = 5


Mounted at /content/drive


In [ ]:
# Load the frozen Scot-BESS split
scotbess_df = pd.read_csv(SPLIT_PATH)

scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)


Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [ ]:
# loading top-k retrieval results
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

val_topk_items = load_jsonl(VAL_TOPK_INPUT)
test_topk_items = load_jsonl(TEST_TOPK_INPUT)

print("Validation top-k items:", len(val_topk_items))
print("Test top-k items:", len(test_topk_items))
print(val_topk_items[0].keys())


Validation top-k items: 165
Test top-k items: 170
dict_keys(['query_id', 'target_text', 'retrieved_examples'])


In [ ]:
# gold labels used only for evaluation
with open(VAL_GOLD_LABELS, "r", encoding="utf-8") as f:
    val_gold_labels = json.load(f)

with open(TEST_GOLD_LABELS, "r", encoding="utf-8") as f:
    test_gold_labels = json.load(f)

print("Validation gold labels:", len(val_gold_labels))
print("Test gold labels:", len(test_gold_labels))

assert len(val_topk_items) == len(scotbess_df_val) == len(val_gold_labels)
assert len(test_topk_items) == len(scotbess_df_test) == len(test_gold_labels)


Validation gold labels: 165
Test gold labels: 170


In [ ]:
# Reuse the same fixed MultiLabelBinarizer used for the Scot-BESS SVC experiment
mlb = joblib.load(MLB_PATH)
ALL_LABELS = list(mlb.classes_)
print("Number of labels:", len(ALL_LABELS))

# Load the fixed Scot-BESS taxonomy definitions
labels_df = pd.read_excel(DEFINITIONS_PATH)

required_definition_columns = {"label", "definition"}
missing = required_definition_columns - set(labels_df.columns)
if missing:
    raise ValueError(f"Definitions file is missing columns: {sorted(missing)}")

labels_df["label"] = labels_df["label"].astype(str).str.strip()
labels_df["definition"] = labels_df["definition"].astype(str).str.strip()

label_descriptions = dict(zip(labels_df["label"], labels_df["definition"]))

# Check that definitions and MLB classes match exactly
assert set(ALL_LABELS) == set(label_descriptions), "Mismatch between MLB labels and taxonomy definitions."

print(labels_df[["label", "definition"]])


Number of labels: 20
                                            label  \
0                            Wildlife and Ecology   
1                                         Traffic   
2                         Fire and Explosion Risk   
3           Landscape, Visual and Heritage Impact   
4      Consultation, Transparency and Information   
5                 Emergency Planning and Response   
6                    Water and Soil Contamination   
7                               Cumulative Impact   
8                                 Light Pollution   
9                                           Noise   
10                           Health and Wellbeing   
11                                 Property Value   
12      Planning Policy and Regulatory Compliance   
13                                 Site Selection   
14                Community and Economic Benefits   
15           Decommissioning and Site Restoration   
16                                   Project Need   
17                       

In [ ]:
OLLAMA_OPTIONS = {
    "temperature": 0,
    "num_predict": 192,
    "num_ctx": 32768, #increased for scotbess
    "seed": SEED}

In [ ]:
def label_block(label_descriptions, labels):
    lines = []
    for label in labels:
        desc = label_descriptions[label]
        lines.append(f" - {label}: {desc}")
    return "\n".join(lines)

LABEL_BLOCK = label_block(label_descriptions, ALL_LABELS)

print(LABEL_BLOCK)
#will be provided in-context
print("Number of labels:", len(ALL_LABELS)) #ok
print(f"Number of words:{len(LABEL_BLOCK.split())}")

 - Agricultural Land: Loss, conversion or degradation of agricultural land, including prime agricultural land, arable or grazing land and food production.
 - Community and Economic Benefits: Community benefit funds, employment, local procurement, economic contributions, or the claimed absence or inadequacy of such benefits.
 - Consultation, Transparency and Information: Public engagement, notification, access to information, incomplete or withheld documents, unanswered formal requests, misleading communication and representation of community feedback. This covers the process of engagement and disclosure, not the technical adequacy of a risk assessment, which should be attributed to the relevant risk label instead.
 - Cumulative Impact: Concerns that the proposal would add to the combined effects of existing, approved, or proposed developments in the surrounding area, creating an excessive concentration or overall burden that is greater than the impact of the project considered in isola

In [ ]:
label_schema = {
    "type": "object",
    "properties": {
        "predicted_labels": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": ALL_LABELS},
        }
    },
    "required": ["predicted_labels"],
    "additionalProperties": False}


In [ ]:
def warmup_ollama():
    """Send a short request to load the model into GPU memory."""
    resp = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": [{"role": "user", "content": "Hi"}],
        "stream": False,
        "options": {"num_predict": 5}
    })
    print("Ollama warmup done:", resp.status_code)

In [ ]:
warmup_ollama()

Ollama warmup done: 200


In [ ]:
#check
print("Validation df length:", len(scotbess_df_val))
print("Validation top-k items:", len(val_topk_items))
print("Test df length:", len(scotbess_df_test))
print("Test top-k items:", len(test_topk_items))

for item in val_topk_items:
    assert item["target_text"] == scotbess_df_val.iloc[int(item["query_id"])]["final_masked_text"]

for item in test_topk_items:
    assert item["target_text"] == scotbess_df_test.iloc[int(item["query_id"])]["final_masked_text"]

print("Retrieval target texts match the frozen Scot-BESS splits.")


Validation df length: 165
Validation top-k items: 165
Test df length: 170
Test top-k items: 170
Retrieval target texts match the frozen Scot-BESS splits.


##Prompt construction

In [ ]:
#The final AAPD-selected retrieval configuration is transferred to Scot-BESS: top-5 demonstrations + label descriptions

In [ ]:
# Scot-BESS text is already natural prose, so no AAPD-specific detokenization is needed
def prepare_text(text):
    return str(text).strip()

def format_retrieved_examples(retrieved_examples, n=N_RETRIEVED):
    example_blocks = []

    for i, ex in enumerate(retrieved_examples[:n], start=1):
        text = prepare_text(ex["text"])
        labels = ex["labels"]

        example_blocks.append(
            f"Example {i}\n"
            f"Planning representation:\n{text}\n"
            f"Labels:\n{json.dumps(labels, ensure_ascii=False)}"
        )

    return "\n\n".join(example_blocks)


##Building prompts

In [ ]:
def build_prompt_with_examples(target_text, retrieved_examples):
    target_text = prepare_text(target_text)
    examples_block = format_retrieved_examples(retrieved_examples)

    system_message = (
        "You are an expert in multi-label topic classification. "
        "Use only the allowed label names and return only the required JSON object.")

    user_message = f"""
You are performing multi-label topic classification for planning representations concerning battery energy storage system developments.
Assign all topic or concern labels that are substantively supported by the target representation. Do not assign labels for incidental mentions.
Use only labels from the allowed label set below.

Allowed labels and descriptions:
{LABEL_BLOCK}

Below are retrieved labelled demonstrations. They show examples of planning representations and their assigned labels.
{examples_block}

Now classify the target planning representation.

Target planning representation:
{target_text}

Return exactly one JSON object in this format:
{{"predicted_labels": ["label1", "label2"]}}

Rules:
- Use only exact label names from the allowed label set.
- Select only labels that are directly and substantively supported by the target representation.
- Use the retrieved demonstrations as guidance, but do not copy their labels unless the target representation itself supports them.
- Return only the JSON object, with no explanation or extra text.

""".strip()

    return [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}]

In [ ]:
def build_topk_messages(item):
    return build_prompt_with_examples(target_text=item["target_text"], retrieved_examples=item["retrieved_examples"])


In [ ]:
def generate_ollama_response(messages):
    total_start = time.time()

    resp = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": messages,
        "format": label_schema,
        "stream": False,
        "options": OLLAMA_OPTIONS
    })
    resp.raise_for_status()
    data = resp.json()

    total_runtime = time.time() - total_start
    raw_output = data["message"]["content"].strip()

    # Ollama returns token counts and timing natively
    input_tokens = data.get("prompt_eval_count")
    output_tokens = data.get("eval_count")
    # eval_duration is in nanoseconds
    eval_ns = data.get("eval_duration", 0)
    generation_runtime = eval_ns / 1e9

    return {
        "response": raw_output,
        "generation_runtime_seconds": generation_runtime,
        "total_sample_runtime_seconds": total_runtime,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }

In [ ]:
def parse_schema_output(raw_output):
    raw_output = str(raw_output).strip()
    parsed = json.loads(raw_output)

    labels = parsed["predicted_labels"]

    if not isinstance(labels, list):
        raise ValueError(f"predicted_labels is not a list: {raw_output}")

    invalid_labels = [
        label for label in labels
        if label not in ALL_LABELS
    ]

    if invalid_labels:
        raise ValueError(f"Invalid labels: {invalid_labels}")

    valid_labels = []
    for label in labels:
        if label not in valid_labels:
            valid_labels.append(label)

    return {
        "predicted_labels": valid_labels,
        "raw_output": raw_output}

In [ ]:
def get_done_query_ids(output_path):
    done = set()

    if not os.path.exists(output_path):
        return done

    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    row = json.loads(line)
                    if row.get("status") == "ok":
                        done.add(row["query_id"])
                except Exception:
                    pass
    return done

In [ ]:
def run_ollama_validation_checkpointed(
    items,
    output_path,
    strategy_name,
    build_messages_fn,
    gold_labels_dict,
    max_items=None):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    if max_items is not None:
        items = items[:max_items]

    done_query_ids = get_done_query_ids(output_path)

    remaining_items = [
        item for item in items
        if item["query_id"] not in done_query_ids]

    print(f"Already completed: {len(done_query_ids)}")
    print(f"Remaining to run: {len(remaining_items)}")

    experiment_start = time.time()

    for item in remaining_items:
        query_id = item["query_id"]
        sample_start = time.time()

        try:
            messages = build_messages_fn(item)

            generation = generate_ollama_response(messages)
            parsed = parse_schema_output(generation["response"])

            row = {
                "status": "ok",
                "strategy": strategy_name,
                "query_id": query_id,
                "gold_labels": gold_labels_dict[str(query_id)],
                "predicted_labels": parsed["predicted_labels"],
                "raw_output": parsed["raw_output"],
                "input_tokens": generation["input_tokens"],
                "output_tokens": generation["output_tokens"],
                "generation_runtime_seconds": generation["generation_runtime_seconds"],
                "total_sample_runtime_seconds": generation["total_sample_runtime_seconds"],
                "wall_time_seconds": time.time() - sample_start,
                "timestamp": datetime.now().isoformat()}

        except Exception as e:
            row = {
                "status": "error",
                "strategy": strategy_name,
                "query_id": query_id,
                "error": repr(e),
                "wall_time_seconds": time.time() - sample_start,
                "timestamp": datetime.now().isoformat()}

        with open(output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"Finished. Total wall time: {(time.time() - experiment_start) / 60:.2f} minutes")

In [ ]:
def load_prediction_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                if row.get("status") == "ok":
                    rows.append(row)

    return rows

def load_prediction_jsonl_all(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))

    return rows


def evaluate_prediction_file(path, mlb=mlb, require_no_errors=True):
    all_rows = load_prediction_jsonl_all(path)

    ok_rows = [row for row in all_rows if row.get("status") == "ok"]
    error_rows = [row for row in all_rows if row.get("status") == "error"]

    if require_no_errors and len(error_rows) > 0:
        raise ValueError(
            f"{path} contains {len(error_rows)} error rows. ")

    y_true_labels = [row["gold_labels"] for row in ok_rows]
    y_pred_labels = [row["predicted_labels"] for row in ok_rows]

    y_true = mlb.transform(y_true_labels)
    y_pred = mlb.transform(y_pred_labels)

    return {
        "path": path,
        "n_total_rows": len(all_rows),
        "n_examples": len(ok_rows),
        "n_error_rows": len(error_rows),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "avg_predicted_labels": np.mean([len(row["predicted_labels"]) for row in ok_rows]),
        "avg_gold_labels": np.mean([len(row["gold_labels"]) for row in ok_rows]),
        "avg_input_tokens": np.mean([row["input_tokens"] for row in ok_rows]),
        "avg_output_tokens": np.mean([row["output_tokens"] for row in ok_rows]),
        "avg_runtime_seconds": np.mean([row["wall_time_seconds"] for row in ok_rows])
    }

##Full validation run

In [ ]:
run_ollama_validation_checkpointed(
    items=val_topk_items,
    output_path=TOPK_VAL_OUTPUT,
    strategy_name="topk_5_examples_label_descriptions",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=val_gold_labels)

Already completed: 0
Remaining to run: 165
Finished. Total wall time: 19.76 minutes


In [ ]:
val_results = evaluate_prediction_file(TOPK_VAL_OUTPUT)
val_results_df = pd.DataFrame([val_results])

val_results_path = f"{OUTPUT_DIR}/SCOTBESS_llama_validation_topk_results.csv"
val_results_df.to_csv(val_results_path, index=False)

assert val_results["n_examples"] == 165

val_results_df

,path,n_total_rows,n_examples,n_error_rows,micro_f1,macro_f1,avg_predicted_labels,avg_gold_labels,avg_input_tokens,avg_output_tokens,avg_runtime_seconds
0,/content/drive/MyDrive/thesis_results/SCOTBESS...,165,165,0,0.718006,0.669991,4.781818,6.157576,5148.278788,34.363636,7.181791


In [ ]:
#check
val_rows = load_prediction_jsonl(TOPK_VAL_OUTPUT)
max_val_input_tokens = max(r["input_tokens"] for r in val_rows if r["input_tokens"] is not None)

print("Maximum validation input tokens:", max_val_input_tokens)
print("Configured context:", OLLAMA_OPTIONS["num_ctx"])

Maximum validation input tokens: 26165
Configured context: 32768


#**Final run on test set**

In [ ]:
run_ollama_validation_checkpointed(
    items=test_topk_items,
    output_path=TOPK_TEST_OUTPUT,
    strategy_name="topk_5_examples_label_descriptions_ollama_schema",
    build_messages_fn=build_topk_messages,
    gold_labels_dict=test_gold_labels,
    max_items=None)


Already completed: 0
Remaining to run: 170
Finished. Total wall time: 20.82 minutes


In [ ]:
test_results = evaluate_prediction_file(TOPK_TEST_OUTPUT)
test_results_df = pd.DataFrame([test_results])


test_results_df

,path,n_total_rows,n_examples,n_error_rows,micro_f1,macro_f1,avg_predicted_labels,avg_gold_labels,avg_input_tokens,avg_output_tokens,avg_runtime_seconds
0,/content/drive/MyDrive/thesis_results/SCOTBESS...,170,170,0,0.7319,0.689541,4.482353,5.917647,5367.482353,32.511765,7.34457


In [ ]:
all_rows = load_prediction_jsonl_all(TOPK_TEST_OUTPUT)

ok_rows = [row for row in all_rows if row.get("status") == "ok"]
error_rows = [row for row in all_rows if row.get("status") == "error"]

if len(error_rows) > 0:
    raise ValueError(f"{TOPK_TEST_OUTPUT} contains {len(error_rows)} error rows.")

llm_y_test_true_labels = [row["gold_labels"] for row in ok_rows]
llm_y_test_pred_labels = [row["predicted_labels"] for row in ok_rows]

llm_y_test = mlb.transform(llm_y_test_true_labels)
llm_y_test_pred = mlb.transform(llm_y_test_pred_labels)

report_dict = classification_report(
    llm_y_test,
    llm_y_test_pred,
    target_names=mlb.classes_,
    zero_division=0,
    output_dict=True)

report_df = pd.DataFrame(report_dict).T
report_df.to_csv(f"{OUTPUT_DIR}/SCOTBESS_llama_Ollama_topk_classification_report.csv")

report_df


,precision,recall,f1-score,support
Agricultural Land,0.812500,0.419355,0.553191,31.0
Community and Economic Benefits,0.720000,0.620690,0.666667,29.0
"Consultation, Transparency and Information",0.827586,0.533333,0.648649,45.0
Cumulative Impact,0.884615,0.489362,0.630137,47.0
Decommissioning and Site Restoration,1.000000,0.421053,0.592593,19.0
Emergency Planning and Response,0.843750,0.529412,0.650602,51.0
Fire and Explosion Risk,0.902174,0.912088,0.907104,91.0
Grid Connection and Electrical Infrastructure,0.800000,0.235294,0.363636,17.0
Health and Wellbeing,0.780000,0.619048,0.690265,63.0
"Landscape, Visual and Heritage Impact",0.886076,0.769231,0.823529,91.0


In [ ]:
total_recorded_wall_time_min = sum(row.get("wall_time_seconds", 0) for row in all_rows) / 60

result = {
    "model": "Llama3.1_8B_Ollama_q4_K_M",
    "dataset": "Scot-BESS",
    "prompt_strategy": "topk_5_examples_label_descriptions",
    "test_f1_micro": f1_score(llm_y_test, llm_y_test_pred, average="micro", zero_division=0),
    "test_f1_macro": f1_score(llm_y_test, llm_y_test_pred, average="macro", zero_division=0),
    "avg_predicted_labels": np.mean([len(row["predicted_labels"]) for row in ok_rows]),
    "avg_inference_time_sec": np.mean([row["total_sample_runtime_seconds"] for row in ok_rows]),
    "inference_per_sample_ms": np.mean([row["total_sample_runtime_seconds"] for row in ok_rows]) * 1000,
    "avg_input_tokens": np.mean([row["input_tokens"] for row in ok_rows if row["input_tokens"] is not None]),
    "max_input_tokens": np.max([row["input_tokens"] for row in ok_rows if row["input_tokens"] is not None]),
    "total_wall_time_min": total_recorded_wall_time_min,
    "n_error_rows": len(error_rows)
}

result_df = pd.DataFrame([result])
result_df.to_csv(f"{OUTPUT_DIR}/SCOTBESS_Llama_Ollama_topk_test_results_summary.csv", index=False)

result


{'model': 'Llama3.1_8B_Ollama_q4_K_M',
 'dataset': 'Scot-BESS',
 'prompt_strategy': 'topk_5_examples_label_descriptions',
 'test_f1_micro': 0.7319004524886877,
 'test_f1_macro': 0.6895410014403379,
 'avg_predicted_labels': np.float64(4.482352941176471),
 'avg_inference_time_sec': np.float64(7.344161443149342),
 'inference_per_sample_ms': np.float64(7344.161443149342),
 'avg_input_tokens': np.float64(5367.482352941176),
 'max_input_tokens': np.int64(27658),
 'total_wall_time_min': 20.809614507357278,
 'n_error_rows': 0}

In [ ]:
# Final context-window check
print("Maximum test input tokens:", result["max_input_tokens"])
print("Configured Ollama context:", OLLAMA_OPTIONS["num_ctx"])



Maximum test input tokens: 27658
Configured Ollama context: 32768


In [ ]:
from google.colab import runtime
runtime.unassign()